# Requirements & Imports

## Requirements

### Python Packages

In [1]:
!pip install -q rpy2

### R Packages

In [2]:
# Habilita o uso das mágicas %R e %%R para rodar código R diretamente em células do notebook.
%load_ext rpy2.ipython

In [3]:
# Instala o pacote R inaparc, que fornece métodos de inicialização para algoritmos de clusterização fuzzy.
%%R
install.packages("inaparc", repos = "https://cloud.r-project.org/")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘kpeaks’, ‘lhs’

trying URL 'https://cloud.r-project.org/src/contrib/kpeaks_1.1.0.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/lhs_1.2.1.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/inaparc_1.2.1.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmpHavfy0/downloaded_packages’


In [4]:
# Instala o pacote R ppclust, que implementa algoritmos de clusterização fuzzy como FCM e GK.
%%R
install.packages("ppclust", repos = "https://cloud.r-project.org/")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cloud.r-project.org/src/contrib/ppclust_1.1.0.1.tar.gz'
Content type 'application/x-gzip' length 3885504 bytes (3.7 MB)
downloaded 3.7 MB


The downloaded source packages are in
	‘/tmp/RtmpHavfy0/downloaded_packages’


## Imports

In [5]:
import os
from dataclasses import dataclass
from typing import Optional, Tuple, Union

import numpy as np
import pandas as pd
import time

from rpy2.robjects import pandas2ri, globalenv, r   # Importa utilitários para converter objetos pandas e acessar o ambiente global do R

# Auxiliary Functions & Helpers

## Helpers

In [6]:
def _build_io_paths(symbol: str, base_input_dir: str, base_output_dir: str) -> dict[str, str]:
    insample_path = os.path.join(
        base_input_dir,
        "insample",
        f"{symbol}_5m_daily_insample.xlsx",
    )
    outsample_path = os.path.join(
        base_input_dir,
        "out_of_sample",
        f"{symbol}_5m_daily_outofsample.xlsx",
    )

    symbol_out_dir = os.path.join(base_output_dir, symbol)
    os.makedirs(symbol_out_dir, exist_ok=True)

    return {
        "insample_path": insample_path,
        "outsample_path": outsample_path,
        "symbol_out_dir": symbol_out_dir,
    }

In [7]:
def _validate_required_columns(
    df: pd.DataFrame,
    required_cols: list[str],
    df_name: str,
) -> None:
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(
            f"{df_name} está sem as colunas obrigatórias: {missing}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )

In [8]:
def _load_and_prepare_base_data(
    insample_path: str,
    outsample_path: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    df_train = pd.read_excel(insample_path)
    df_test = pd.read_excel(outsample_path)

    # Mantém o comportamento das funções originais:
    # tenta renomear "Data" -> "Date" caso exista.
    if "Data" in df_train.columns:
        df_train.rename(columns={"Data": "Date"}, inplace=True)
    if "Data" in df_test.columns:
        df_test.rename(columns={"Data": "Date"}, inplace=True)

    _validate_required_columns(df_train, ["target", "RV"], "df_train")
    _validate_required_columns(df_test, ["target", "RV"], "df_test")

    # Se quiser exigir Date também, descomente:
    # _validate_required_columns(df_train, ["Date"], "df_train")
    # _validate_required_columns(df_test, ["Date"], "df_test")

    df_train["Target"] = df_train["target"]
    df_test["Target"] = df_test["target"]

    df_train["Feature Origin"] = df_train["RV"]
    df_test["Feature Origin"] = df_test["RV"]

    return df_train, df_test


In [9]:
def _save_outputs(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    symbol: str,
    horizon: int,
    symbol_out_dir: str,
) -> tuple[str, str]:
    train_out_path = os.path.join(
        symbol_out_dir,
        f"{symbol}_tplus{horizon}_train_with_fuzzy.xlsx",
    )
    test_out_path = os.path.join(
        symbol_out_dir,
        f"{symbol}_tplus{horizon}_test_with_fuzzy.xlsx",
    )

    df_train.to_excel(train_out_path, index=False)
    df_test.to_excel(test_out_path, index=False)

    return train_out_path, test_out_path

In [10]:
@dataclass(frozen=True)
class HorizonConfig:
    horizon: int
    base_input_dir: str
    base_output_dir: str


def _build_default_horizon_config() -> dict[int, HorizonConfig]:
    """
      Monta o mapa de configuração padrão por horizonte.
    """
    return {
        1: HorizonConfig(
            horizon=1,
            base_input_dir=input_RV_t1,
            base_output_dir=output_RV_t1,
        ),
        7: HorizonConfig(
            horizon=7,
            base_input_dir=input_RV_t7,
            base_output_dir=output_RV_t7,
        ),
        30: HorizonConfig(
            horizon=30,
            base_input_dir=input_RV_t30,
            base_output_dir=output_RV_t30,
        ),
    }


## Functions for data preprocessing

In [11]:
def _normalize_dates(
    df: pd.DataFrame
) -> pd.DataFrame:

    """
      Description:
        Converte a coluna 'Date' do dataframe recebido como
        parâmetro (assumindo que essa coluna existe) para datetime.

      Params:
        df (pd.DataFrame): DataFrame de entrada contendo a coluna 'Date',
          além de 'Feature Origin' e 'Target'. Assume-se que as linhas já
          chegam ordenadas por data.

      Return:
        df (pd.DataFrame): Cópia do DataFrame original com a coluna 'Date'
          convertida para datetime (valores inválidos viram NaT).
    """

    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    return df

In [12]:
def _align_y_next(
    target: pd.Series
) -> pd.Series:

    """
      Description:
        Prepara o alvo para previsão one-step-ahead. Ao deslocar a série em -1,
        garantimos que as features calculadas em t sejam pareadas com o valor
        de Target observado em t+1 (o “próximo período”). Isso evita vazamento
        de informação e alinha corretamente X_t -> y_{t+1} no treinamento/teste.

      Params:
        target (pd.Series): Série alvo original (Target_t) no mesmo índice
          temporal das features.

      Return:
        (pd.Series): Série alvo deslocada em -1 (Target_{t+1}); a última
          posição torna-se NaN por não haver observação futura correspondente.
    """

    return target.shift(-1)

In [13]:
def make_har_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    week_len: int = 5,
    month_len: int = 22,
    include_daily: bool = True,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Constrói as features HAR (Heterogeneous Autoregressive) a partir da coluna
    'Feature Origin' e gera o alvo deslocado em um passo à frente:

        y(t) = Target_{t+1}

    Retorna:
        X_train, y_train, X_test, y_test
    """

    if week_len < 2:
        raise ValueError("week_len deve ser >= 2.")
    if month_len < week_len:
        raise ValueError("month_len deve ser >= week_len.")

    # 1. Normaliza datas e copia
    tr = _normalize_dates(train_df)
    te = _normalize_dates(test_df)

    # 2. Concatena treino + teste numa série temporal contínua
    full = pd.concat([tr, te], ignore_index=True)

    n_tr = len(tr)
    n_full = len(full)

    # 3. Extrai a série base de volatilidade
    fo_all = full["Feature Origin"].to_numpy(dtype=float)
    s_all = pd.Series(fo_all)

    # 4. Cria as colunas HAR sobre a série completa
    feats_all = []

    if include_daily:
        feats_all.append(s_all.to_numpy())

    week_col = (
        s_all.rolling(window=week_len, min_periods=week_len)
        .mean()
        .to_numpy()
    )
    feats_all.append(week_col)

    month_col = (
        s_all.rolling(window=month_len, min_periods=month_len)
        .mean()
        .to_numpy()
    )
    feats_all.append(month_col)

    X_all = np.vstack(feats_all).T

    # 5. Alvo deslocado em um passo à frente
    y_all_full = _align_y_next(full["Target"])

    # 6. Linhas matematicamente válidas
    valid_all = (~np.isnan(X_all).any(axis=1)) & (~y_all_full.isna().to_numpy())

    # 7. Define treino/teste com base em onde cai o alvo t+1
    idx_next = np.arange(n_full) + 1
    mask_y_in_train = idx_next < n_tr
    mask_y_in_test = (idx_next >= n_tr) & (idx_next < n_full)

    valid_train = valid_all & mask_y_in_train
    valid_test = valid_all & mask_y_in_test

    # 8. Separa os conjuntos finais
    X_train = X_all[valid_train].astype(np.float64)
    y_train = y_all_full[valid_train].to_numpy(dtype=np.float64)

    X_test = X_all[valid_test].astype(np.float64)
    y_test = y_all_full[valid_test].to_numpy(dtype=np.float64)

    return X_train, y_train, X_test, y_test

## Functions for training and testing model

In [14]:
def gaussmf(
    x: Union[float, np.ndarray],
    c: float,
    s: float
) -> Union[float, np.ndarray]:

    """
        Description:
            Calcula o grau de pertinência usando uma função de pertinência gaussiana.

        Args:
            x (float | np.ndarray): Valor(es) de entrada para calcular o grau de pertinência.
            c (float): Centro da função gaussiana.
            s (float): Desvio padrão da função gaussiana.

        Return:
            float | np.ndarray: Grau de pertinência calculado para o(s) valor(es) de entrada.
    """

    aux = (x - c)/s  # Calcula a distância normalizada do valor ao centro
    z = np.exp(-(aux*aux)/2)  # Aplica a fórmula da função gaussiana

    return z

In [15]:
def gaussgranules(
    x: np.ndarray,
    centers: np.ndarray,
    sigmas: np.ndarray
) -> np.ndarray:

    """
        Description:
            Calcula as ativações de múltiplas regras fuzzy usando funções gaussianas.

        Args:
            x (np.ndarray): Vetor de entrada de dimensão d.
            centers (np.ndarray): Array de formato (N, d) com os centros das N regras.
            sigmas (np.ndarray): Array de formato (N, d) com os desvios padrão das N regras.

        Return:
            np.ndarray: Array de formato (N,) contendo o grau de ativação para cada uma das N regras.
    """

    # Um centro c = (c_{1},c_{2},...,c_{n}) e um desvio padrão s = (s_{1},s_{2},...,s_{n}) definem uma regra fuzzy (chamemos r).
    # Quando aplicamos gaussmf(x,c,s) para x = (x_{1},x_{2},...,x_{n}) obtemos o grau de pertencimento do vetor x à regra r.

    N, d = centers.shape  # N é o número de regras, d é a dimensão representada pelo delay
    activations = []  # Lista para armazenar as ativações de cada regra

    for i in range(N):
        mu = 1.0  # Inicializa o grau de pertinência como 1
        for j in range(d):
            # Multiplica o grau de pertinência pela ativação da j-ésima dimensão
            mu *= gaussmf(x[j], centers[i, j], sigmas[i, j]) # Multiplica o grau de pertinência da dimensão j de x à regra i
        activations.append(mu)  # Adiciona o resultado à lista de ativações

    return np.array(activations)  # Converte a lista para um array numpy

In [16]:
%%R

#' @title Clusterização Fuzzy com Gustafson-Kessel
#' @description
#' Executa o algoritmo de clusterização fuzzy Gustafson–Kessel (GK) com inicialização por k-means++ e pertinências fuzzy aleatórias.
#' A função também calcula, manualmente, as matrizes de dispersão (\eqn{\Sigma_j}) e os desvios padrão por dimensão para cada cluster.
#'
#' @param x Um data.frame ou matriz numérica com n amostras (linhas) e d variáveis (colunas).
#' @param k Um inteiro representando o número de clusters (ou regras fuzzy).
#'
#' @return Uma lista com três elementos:
#' \describe{
#'   \item{\code{centros}}{Matriz \code{k x d} contendo os centros finais de cada cluster.}
#'   \item{\code{pertinencias}}{Matriz \code{n x k} com os graus de pertinência fuzzy das amostras aos clusters.}
#'   \item{\code{desvios}}{Lista de \code{k} matrizes \code{d x d}, cada uma representando a matriz de dispersão ponderada (\eqn{\Sigma_j}) de um cluster.}
#' }
#'
#' @details
#' A função usa:
#' \itemize{
#'   \item \code{ppclust::gk} para a execução do algoritmo GK;
#'   \item \code{inaparc::kmpp} para inicialização dos centros;
#'   \item \code{inaparc::imembrand} para inicialização da matriz de pertinência.
#' }
#' O expoente de fuzzificação \code{m} é fixado em 2. As matrizes de dispersão são calculadas usando os pesos fuzzy elevados a \code{m}, conforme a definição clássica.
#'

gk_clustering <- function(x, k) {
  # Carrega o pacote 'ppclust', que fornece implementações de algoritmos de clusterização fuzzy,
  # como Fuzzy C-Means (FCM), Gustafson–Kessel (GK), entre outros.
  library(ppclust)

  # Carrega o pacote 'inaparc', que provê métodos de inicialização de centros e pertinências,
  # como k-means++ (kmpp) e inicialização fuzzy aleatória (imembrand).
  library(inaparc)

  # Garante que k seja um escalar inteiro válido
  k <- as.integer(as.numeric(k[1]))

  # ----------- Inicialização dos clusters -----------

  # Semente para reprodutibilidade de resultados
  set.seed(123)

  # Inicializa os centros dos clusters com o algoritmo k-means++.
  # Retorna uma lista com vários componentes; o '$v' acessa a matriz dos centros inicializados (k x d).
  v <- kmpp(x, k = k)$v

  # Inicializa a matriz de pertinência fuzzy (n x k), onde cada linha soma 1.
  # Essa matriz define o grau de associação inicial de cada amostra a cada cluster.
  u <- imembrand(nrow(x), k = k)$u

  # ----------- Execução do algoritmo Gustafson-Kessel -----------

  # Executa o algoritmo GK (fuzzy clustering com métricas de covariância adaptativas).
  # Argumentos principais:
  # - x: conjunto de dados
  # - centers: centros inicializados (matriz k x d)
  # - memberships: matriz de pertinência inicial (n x k)
  # - m: expoente de fuzzificação (m > 1; m = 2 é o padrão mais usado)
  # - dmetric: métrica de distância ("sqeuclidean" = euclidiana ao quadrado)
  # - iter.max: número máximo de iterações
  # - con.val: critério de convergência (parar se mudança entre iterações < 1e-9)
  gk.res <- gk(x, centers = v, memberships = u, m = 2,
              dmetric = "sqeuclidean", iter.max = 1000, con.val = 1e-9)

  # Exporta a matriz de centros finais (k x d) para uso posterior
  centros <- gk.res$v

  # Exporta a matriz final de pertinência fuzzy (n x k)
  pertinencias <- gk.res$u

  # ----------- Cálculo manual das matrizes de dispersão (Σ_j) e desvios padrão -----------

  # Converte os dados para uma matriz numérica (n x d), onde:
  # n = número de amostras (linhas), d = número de variáveis (lags ou dimensões)
  X <- as.matrix(x)

  # Recupera a matriz de pertinência fuzzy (n x k), onde:
  # U[i, j] representa o grau de associação da amostra i ao cluster j
  U <- gk.res$u

  # Recupera a matriz de centros dos clusters (k x d), onde:
  # V[j, ] contém as coordenadas do centro do cluster j
  V <- gk.res$v

  # Expoente de fuzzificação usado tanto no algoritmo quanto neste cálculo
  m <- 2

  # Define as dimensões principais do problema
  n <- nrow(X)   # número de amostras
  k <- ncol(U)   # número de clusters
  d <- ncol(X)   # número de variáveis (dimensões)

  # Inicializa uma lista que armazenará a matriz de dispersão Σ_j de cada cluster j
  Sigma_list <- vector("list", k)

  # Loop principal: calcula Σ_j para cada cluster j
  for (j in 1:k) {

    # Inicializa o numerador e denominador da fórmula de Σ_j
    # Σ_j = ( ∑_{i=1}^{n} (u_ij)^m * (x_i - v_j)(x_i - v_j)^T ) / ∑_{i=1}^{n} (u_ij)^m
    num <- matrix(0, d, d)  # acumulador da soma ponderada das covariâncias
    denom <- 0              # acumulador da soma dos pesos (u_ij^m)

    # Loop interno: percorre todas as amostras i
    for (i in 1:n) {

      # Calcula o grau de pertinência fuzzy elevado ao expoente m
      u_ij_m <- U[i, j]^m

      # Calcula o vetor de diferença entre a amostra i e o centro do cluster j:
      # diff = x_i - v_j (vetor coluna d x 1)
      diff <- matrix(X[i, ] - V[j, ], ncol = 1)

      # Atualiza o numerador com a contribuição da amostra i:
      # u_ij^m * (diff * diffᵗ) → matriz d x d (produto externo)
      num <- num + u_ij_m * (diff %*% t(diff))

      # Atualiza o denominador com o peso u_ij^m
      denom <- denom + u_ij_m
    }

    # Finaliza o cálculo da matriz de dispersão Σ_j:
    # divide o somatório ponderado das covariâncias pelo total de pesos
    Sigma_list[[j]] <- num / denom
  }

  # Exporta os desvios padrão com o nome 'desvios'
  desvios <- Sigma_list

  return(list(
    centros = centros,
    pertinencias = pertinencias,
    desvios = desvios
  ))

}

In [17]:
def run_gk_clustering(
    X_train: pd.DataFrame,
    rules_number: int
    ) -> Tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:

    """
      Description:
          Executa a função R 'gk_clustering' para realizar clusterização fuzzy com o algoritmo
          Gustafson–Kessel. Retorna os centros dos clusters, as pertinências fuzzy e os desvios
          padrão por dimensão de cada cluster, extraídos das matrizes de covariância.

      Args:
          X_train (pd.DataFrame): Conjunto de dados com colunas defasadas (ex: 'lag1', 'lag2', ...).
          rules_number (int): Número de clusters ou regras fuzzy (k).

      Return:
          Tuple contendo:
              - centers_df (pd.DataFrame): Matriz (k x d) com os centros finais dos clusters.
              - memberships_df (pd.DataFrame): Matriz (n x k) de pertinência fuzzy das amostras.
              - sigmas (np.ndarray): Matriz (k x d) com os desvios padrão por dimensão para cada cluster.
    """

    # Garante que Xtrain seja um DataFrame, mesmo que seja um array
    X_train = pd.DataFrame(X_train) if not isinstance(X_train, pd.DataFrame) else X_train

    # Se quiser, nomeie automaticamente as colunas como lag1, lag2, ..., lagN
    if X_train.columns.dtype == 'int':
        X_train.columns = [f"lag{i+1}" for i in range(X_train.shape[1])]

    pandas2ri.activate()  # Ativa a conversão automática entre pandas DataFrame (Python) e data.frame (R)

    # Envia o DataFrame Xtrain para o ambiente global do R com o nome 'x'
    globalenv["x"] = pandas2ri.py2rpy(X_train)

    # Envia o número de clusters/regras para o R como vetor inteiro, com o nome 'rules_number'
    globalenv["k"] = int(rules_number)

    # Executa a função R declarada anteriormente
    r_result = r("gk_clustering(x, k[[1]])")

    centers = r_result.rx2("centros")
    memberships = r_result.rx2("pertinencias")
    desvios = r_result.rx2("desvios")

    centers_df = pd.DataFrame(centers, columns=X_train.columns)
    memberships_df = pd.DataFrame(memberships)
    std_devs = [np.array(desvios[i]) for i in range(len(desvios))]

    # Converte cada matriz R da lista para um array NumPy
    # Cada elemento da lista representa a matriz de covariância (d x d) de um cluster
    sigmas = [np.array(dev) for dev in desvios]

    # Extrai os desvios padrão (raiz quadrada das variâncias) de cada matriz de covariância
    # Isso gera uma lista de vetores 1D, onde cada vetor contém os desvios padrão por dimensão
    sigmas = [np.sqrt(np.diag(S)) for S in sigmas]

    # Concatena a lista de vetores em um array 2D (n_clusters x n_features)
    # Isso facilita o acesso via sigmas[i, j] para a dimensão j do cluster i
    sigmas = np.array(sigmas)  # shape: (n_clusters, n_features)

    return centers_df, memberships_df, sigmas

In [18]:
def train_model(
    Xtrain: np.ndarray,
    ytrain: np.ndarray,
    r: int,
    lamb: float,
    alfa: float,
    ze: float,                 # <- mudou: agora recebemos apenas ze
    centers: np.ndarray,
    sigmas: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:

    """
      Treina um modelo ALSM com atualização online dos parâmetros.
      Usa ativações gaussianas e fator de confiança adaptativo psi.

      Parâmetros do kernel (derivados internamente):
        kc é constante (1/sqrt(2π));
        kz = kc / (ze^3);
        zs = 1 / (2 * ze^2).
    """

    # --- kc fixo e derivação de kz/zs a partir de ze (mantém compatibilidade conceitual) ---
    kc = 1.0 / np.sqrt(2.0 * np.pi)
    kz = kc / (ze ** 3)
    zs = 1.0 / (2.0 * (ze ** 2))

    ndatatrain = Xtrain.shape[0]
    OT = np.zeros(ndatatrain)
    norm = np.zeros(ndatatrain)

    P = alfa * np.eye(2 * r)
    u = np.zeros(2 * r)
    a = np.zeros(2 * r)

    # Histórico (inalterado no formato; agora registra ze, kz e zs calculados)
    hist: Dict[str, Any] = {
        "n": int(ndatatrain),
        "n_features": int(Xtrain.shape[1]),
        "params": {"r": r, "lamb": lamb, "alfa": alfa, "ze": ze, "kz": kz, "zs": zs},
        "y_true":      np.zeros(ndatatrain, dtype=float),
        "y_pred_pre":  np.zeros(ndatatrain, dtype=float),
        "error":       np.zeros(ndatatrain, dtype=float),
        "psi":         np.zeros(ndatatrain, dtype=float),
        "aTPa":        np.zeros(ndatatrain, dtype=float),
        "b":           np.zeros(ndatatrain, dtype=float),
        "sumd":        np.zeros(ndatatrain, dtype=float),
        "d_min":       np.zeros(ndatatrain, dtype=float),
        "d_max":       np.zeros(ndatatrain, dtype=float),
        "a_norm":      np.zeros(ndatatrain, dtype=float),
        "u_norm":      np.zeros(ndatatrain, dtype=float),
        "P_trace":     np.zeros(ndatatrain, dtype=float),
        "P_cond":      np.zeros(ndatatrain, dtype=float),
    }

    # -------- Salvaguardas numéricas (constantes) ----------
    EPS    = 1e-300  # piso para denominadores (~zero, mas não zero) para evitar divisão por zero.
                     # Impacto: se valores normais, nunca é usado; se sumd≈0, evita NaN e torna a atualização quase nula.
    BMIN   = 1e-300  # piso para o denominador b (evita divisão por um número tão pequeno que causaria estouro).
    LMIN   = 1e-12   # piso mínimo para λ na divisão de P (se alguém passar λ muito pequeno/acidentalmente zero).
    ZSMIN  = 1e-12   # piso para zs (evita divisão por zero/underflow em exp(-sq/zs)).
    EXP_CLIP = -700  # limiar para cortar o expoente da exponencial (np.exp(-700) ~ 5e-305, abaixo disso vira 0 sob precisão dupla).

    st = time.process_time()

    for t in range(ndatatrain):
        x = Xtrain[t]

        # --- Ativações gaussianas ---
        # Saneamento numérico: converte d para float e remove NaN/Inf por 0 (pertinência nula).
        # Justificativa: se gaussgranules retornar algo inválido por escala extrema, evitamos contaminar o vetor 'a'.
        d = np.asarray(gaussgranules(x, centers, sigmas), dtype=float).ravel()
        d = np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0)

        # Soma das ativações (denominador da normalização)
        sumd = float(np.sum(d))
        # Se sumd não for finito ou for <= 0, usamos EPS (quase zero) para impedir divisão por zero.
        # Isso preserva as razões (se existirem) e, quando tudo é 0, gera um 'a' de magnitude minúscula,
        # levando a uma atualização praticamente nula (comportamento seguro).
        if (not np.isfinite(sumd)) or (sumd <= 0.0):
            sumd = EPS

        # Vetor de regressão 'a' com normalização segura por sumd
        for j in range(r):
            a[2 * j]     = d[j]**2 / sumd
            a[2 * j + 1] = d[j]    / sumd

        # Predição antes de atualizar (para medir o erro)
        y_pre = float(np.dot(a, u))
        erro  = float(ytrain[t] - y_pre)
        sq    = float(np.abs(erro)**2)

        # --- Psi com proteção ---
        # zs_eff garante zs > 0; exp_arg é cortado para evitar underflow extremo em np.exp.
        # Quando exp_arg < -700, definimos exp como 0.0 explicitamente (equivalente numérico).
        zs_eff  = max(float(zs), ZSMIN)
        exp_arg = -sq / zs_eff
        psi     = float(kz) * (0.0 if exp_arg < EXP_CLIP else np.exp(exp_arg))

        # --- Ganho RLS com proteção ---
        aTPa = float(np.dot(np.dot(a, P), a))
        b    = float(lamb + psi * aTPa)

        Pa = np.dot(P, a)

        # lambda_eff evita divisão por λ muito pequeno; b é verificado quanto a finitude e piso mínimo.
        lambda_eff = max(float(lamb), LMIN)
        if (not np.isfinite(b)) or (abs(b) < BMIN):
            # Atualização é pulada nesta iteração para evitar corromper P e u com números inválidos.
            # Impacto: mínimo — apenas quando a configuração numérica está patológica; caso normal, não dispara.
            pass
        else:
            P = (P - psi * np.outer(Pa, Pa) / b) / lambda_eff
            u = u + (psi * np.dot(P, a) * erro)

        # Saídas pós-atualização
        OT[t]   = np.dot(a, u)
        norm[t] = np.linalg.norm(u)

        # Logs/histórico (inalterado)
        hist["y_true"][t]     = float(ytrain[t])
        hist["y_pred_pre"][t] = y_pre
        hist["error"][t]      = erro
        hist["psi"][t]        = float(psi)
        hist["aTPa"][t]       = aTPa
        hist["b"][t]          = float(b)
        hist["sumd"][t]       = float(np.sum(d))
        hist["d_min"][t]      = float(np.min(d))
        hist["d_max"][t]      = float(np.max(d))
        hist["a_norm"][t]     = float(np.linalg.norm(a))
        hist["u_norm"][t]     = float(norm[t])
        hist["P_trace"][t]    = float(np.trace(P))
        try:
            hist["P_cond"][t] = float(np.linalg.cond(P))
        except Exception:
            # Se P ficar singular/indefinido numericamente, registramos ∞ (evita crash do log).
            hist["P_cond"][t] = np.inf

    et = time.process_time()
    train_time = et - st

    # Fechamento do histórico
    hist["train_time_seconds"] = float(train_time)
    hist["u_final"] = u.copy()
    hist["P_final"] = P.copy()
    hist["OT"]      = OT.copy()
    hist["norm"]    = norm.copy()

    train_model._last_history = hist

    return OT, norm, u, P


In [19]:
def test_model(
    Xtest: np.ndarray,
    ytest: np.ndarray,
    r: int,
    lamb: float,
    P: np.ndarray,
    u: np.ndarray,
    centers: np.ndarray,
    sigmas: np.ndarray
) -> np.ndarray:

    """
      Realiza a predição adaptativa com o modelo ALSM usando atualização online no teste.
    """

    ndatatest = Xtest.shape[0]
    OS = np.zeros(ndatatest)
    a  = np.zeros(2 * r)

    # Histórico compatível com o do treino
    hist: Dict[str, Any] = {
        "n": int(ndatatest),
        "n_features": int(Xtest.shape[1]),
        "params": {"r": r, "lamb": lamb, "alfa": None, "ze": None, "kz": None, "zs": None},
        "y_true":      np.zeros(ndatatest, dtype=float),
        "y_pred_pre":  np.zeros(ndatatest, dtype=float),
        "error":       np.zeros(ndatatest, dtype=float),
        "psi":         np.zeros(ndatatest, dtype=float),
        "aTPa":        np.zeros(ndatatest, dtype=float),
        "b":           np.zeros(ndatatest, dtype=float),
        "sumd":        np.zeros(ndatatest, dtype=float),
        "d_min":       np.zeros(ndatatest, dtype=float),
        "d_max":       np.zeros(ndatatest, dtype=float),
        "a_norm":      np.zeros(ndatatest, dtype=float),
        "u_norm":      np.zeros(ndatatest, dtype=float),
        "P_trace":     np.zeros(ndatatest, dtype=float),
        "P_cond":      np.zeros(ndatatest, dtype=float),
    }

    # Salvaguardas numéricas do teste (análogas às do treino)
    EPS  = 1e-300   # piso para sumd
    BMIN = 1e-300   # piso para b
    LMIN = 1e-12    # piso para λ

    st = time.process_time()

    for t in range(ndatatest):
        x = Xtest[t]

        # Ativações gaussianas e saneamento
        d = np.asarray(gaussgranules(x, centers, sigmas), dtype=float).ravel()
        d = np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0)

        sumd = float(np.sum(d))
        # Se todas as ativações forem quase-zero (ou inválidas), usa EPS para evitar divisão por zero.
        # Isso faz 'a' ficar de magnitude minúscula, provocando atualização desprezível — seguro e coerente.
        if (not np.isfinite(sumd)) or (sumd <= 0.0):
            sumd = EPS

        # Vetor 'a' normalizado com proteção
        for j in range(r):
            a[2 * j]     = d[j]**2 / sumd
            a[2 * j + 1] = d[j]    / sumd

        # Predição e erro (pré-atualização)
        y_pre = float(np.dot(a, u))
        err   = float(ytest[t] - y_pre)

        # Ganho RLS com proteção
        aTPa = float(np.dot(np.dot(a, P), a))
        b    = float(lamb + aTPa)
        Pa   = np.dot(P, a)

        lambda_eff = max(float(lamb), LMIN)
        if (not np.isfinite(b)) or (abs(b) < BMIN):
            # Pula a atualização nesta iteração (evita contaminar P/u com números ruins).
            pass
        else:
            P = (P - np.outer(Pa, Pa) / b) / lambda_eff
            u = u + (np.dot(P, a) * (ytest[t] - np.dot(a, u)))

        OS[t] = np.dot(a, u)

        # Logs da iteração
        hist["y_true"][t]     = float(ytest[t])
        hist["y_pred_pre"][t] = y_pre
        hist["error"][t]      = err
        hist["psi"][t]        = 0.0            # no teste não há psi (mantido por compatibilidade de log)
        hist["aTPa"][t]       = aTPa
        hist["b"][t]          = b
        hist["sumd"][t]       = float(np.sum(d))
        hist["d_min"][t]      = float(np.min(d))
        hist["d_max"][t]      = float(np.max(d))
        hist["a_norm"][t]     = float(np.linalg.norm(a))
        hist["u_norm"][t]     = float(np.linalg.norm(u))
        hist["P_trace"][t]    = float(np.trace(P))
        try:
            hist["P_cond"][t] = float(np.linalg.cond(P))
        except Exception:
            hist["P_cond"][t] = np.inf

    et = time.process_time()

    # Finaliza o histórico (mesmas chaves do treino)
    hist["train_time_seconds"] = float(et - st)
    hist["u_final"] = u.copy()
    hist["P_final"] = P.copy()
    hist["OT"]      = OS.copy()
    hist["norm"]    = hist["u_norm"].copy()

    test_model._last_history = hist

    return OS

## Hyperparameter optimization functions

In [20]:
def _rmse(y_true, y_pred):
    if y_true.size == 0:
        return float("inf")
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

In [21]:
def _mae(y_true, y_pred):
    if y_true.size == 0:
        return float("inf")
    return float(np.mean(np.abs(y_true - y_pred)))

In [22]:
def grid_search_rules_features(
    df_train,
    df_test,
    rules_grid,
    *,
    lamb,
    alfa,
    ze,
    har_week_len: int = 7,
    har_month_len: int = 30,
    har_include_daily: bool = True,
    verbose: bool = True,
):

    ze = float(ze)

    wl = int(har_week_len)
    ml = max(wl, int(har_month_len))

    X_tr, y_tr, _, _ = make_har_features(
        df_train,
        df_test,
        week_len=wl,
        month_len=ml,
        include_daily=har_include_daily
    )

    if X_tr.size == 0 or X_tr.shape[1] == 0:
        raise RuntimeError("HAR: conjunto de treino vazio com as janelas fornecidas.")

    trials = []
    best = None

    for r_rules in rules_grid:
        try:
            centers_df, memberships_df, sigmas = run_gk_clustering(X_tr, int(r_rules))

            OT, norm, u, P = train_model(
                X_tr,
                y_tr,
                int(r_rules),
                lamb,
                alfa,
                ze,
                centers_df.to_numpy(),
                sigmas,
            )
        except Exception as e:
            if verbose:
                print(f"[skip] mode=har r={r_rules} ze={ze:.6f}: treino -> {e}")
            continue

        rmse_tr = _rmse(y_tr, OT)
        mae_tr = _mae(y_tr, OT)

        trial = {
            "feature_mode": "har",
            "r": int(r_rules),
            "delay_like": 3,
            "ze": ze,
            "n_train": int(X_tr.shape[0]),
            "n_features": int(X_tr.shape[1]),
            "rmse_train": rmse_tr,
            "mae_train": mae_tr,
            "OT": OT,
            "norm": norm,
            "u": u,
            "P": P,
            "centers": centers_df.to_numpy(),
            "sigmas": sigmas,
        }

        trials.append(trial)

        if verbose:
            print(
                f"[trial] mode=har r={r_rules:>2d} ze={ze:.6f} "
                f"| n={X_tr.shape[0]:>4d} "
                f"| RMSE={rmse_tr:.6f} | MAE={mae_tr:.6f}"
            )

        if best is None or trial["rmse_train"] < best["rmse_train"]:
            best = trial

    if best is None:
        raise RuntimeError("Nenhum experimento válido foi executado.")

    if verbose:
        print(
            f"[best] mode=har r={best['r']} ze={best['ze']:.6f} "
            f"| RMSE={best['rmse_train']:.6f}"
        )

    return {
        "best": {
            "feature_mode": best["feature_mode"],
            "r": best["r"],
            "delay_like": best["delay_like"],
            "ze": best["ze"],
            "rmse_train": best["rmse_train"],
            "mae_train": best["mae_train"],
            "u": best["u"],
            "P": best["P"],
            "centers": best["centers"],
            "sigmas": best["sigmas"],
            "n_train": best["n_train"],
            "n_features": best["n_features"],
        },
        "trials": trials,
    }

In [23]:
def _select_best_har_hyperparameters(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    symbol_out_dir: str,
    rules_grid: list[int],
    ze: float,
    lamb: float,
    alfa: float,
    week_len: int,
    month_len: int,
) -> tuple[int, float]:
    res_har = grid_search_rules_features(
        df_train=df_train,
        df_test=df_test,
        rules_grid=rules_grid,
        lamb=lamb,
        alfa=alfa,
        ze=ze,
        har_week_len=week_len,
        har_month_len=month_len,
        har_include_daily=True,
        verbose=False,
    )

    best_rules_number_har = int(res_har["best"]["r"])
    best_ze_har = float(res_har["best"]["ze"])

    return best_rules_number_har, best_ze_har

In [24]:
def _run_best_har_block(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    best_rules_number_har: int,
    best_ze_har: float,
    lamb: float,
    alfa: float,
    week_len: int,
    month_len: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    X_train_har, y_train_har, X_test_har, y_test_har = make_har_features(
        train_df=df_train,
        test_df=df_test,
        week_len=week_len,
        month_len=month_len,
        include_daily=True,
    )

    centers_df_har, memberships_df_har, sigmas_har = run_gk_clustering(
        X_train_har,
        best_rules_number_har,
    )

    OT_har, norm_har, u_har, P_har = train_model(
        X_train_har,
        y_train_har,
        best_rules_number_har,
        lamb,
        alfa,
        best_ze_har,
        centers_df_har.to_numpy(),
        sigmas_har,
    )

    OS_har = test_model(
        X_test_har,
        y_test_har,
        best_rules_number_har,
        lamb,
        P_har,
        u_har,
        centers_df_har.to_numpy(),
        sigmas_har,
    )

    df_train = df_train.copy()
    df_test = df_test.copy()

    df_train["Fuzzy HAR"] = np.nan
    start_idx_har = month_len

    df_train.loc[
        df_train.index[start_idx_har : start_idx_har + len(OT_har)],
        "Fuzzy HAR",
    ] = OT_har

    df_test["Fuzzy HAR"] = OS_har

    return df_train, df_test

# Model Train & Model Test

## Connect to Google Drive

In [25]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Parameter Setting

In [26]:
# ----------------------------
# Caminhos por horizonte
# ----------------------------

#
input_RV_t1 = "/content/drive/MyDrive/RVs/RV_t+1"
#
output_RV_t1 = "/content/drive/MyDrive/ProductionFuzzyHARPipeline/RV_t+1"

#
input_RV_t7 = "/content/drive/MyDrive/RVs/RV_t+7"
#
output_RV_t7 = "/content/drive/MyDrive/ProductionFuzzyHARPipeline/RV_t+7"

#
input_RV_t30 = "/content/drive/MyDrive/RVs/RV_t+30"
#
output_RV_t30 = "/content/drive/MyDrive/ProductionFuzzyHARPipeline/RV_t+30"


# ----------------------------
# Grades de busca
# ----------------------------

#
rules_grid = [2, 3, 4, 5, 6]
#
ze_grid = [1]


# ----------------------------
# Parâmetros do Fuzzy HAR
# ----------------------------

#
week_len = 7
#
month_len = 30
#
lamb = 0.99
#
alfa = 1000
#
ze = 1

# ----------------------------
# Mapa de horizontes
# ----------------------------

#
HORIZON_CONFIG = _build_default_horizon_config()

## Run Model

In [27]:
def run_fuzzy_for_symbol(symbol: str, horizon: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Roda apenas o pipeline fuzzy HAR para um símbolo e horizonte.
    """
    if horizon not in HORIZON_CONFIG:
        raise ValueError(f"Horizonte inválido: {horizon}. Use 1, 7 ou 30.")

    config = HORIZON_CONFIG[horizon]

    # 1) Paths
    paths = _build_io_paths(
        symbol=symbol,
        base_input_dir=config.base_input_dir,
        base_output_dir=config.base_output_dir,
    )

    # 2) Carrega e prepara os dados
    df_train, df_test = _load_and_prepare_base_data(
        insample_path=paths["insample_path"],
        outsample_path=paths["outsample_path"],
    )

    # 3) Seleção de hiperparâmetros apenas para HAR
    best_rules_number_har, best_ze_har = _select_best_har_hyperparameters(
        df_train=df_train,
        df_test=df_test,
        symbol_out_dir=paths["symbol_out_dir"],
        rules_grid=rules_grid,
        ze=ze,
        lamb=lamb,
        alfa=alfa,
        week_len=week_len,
        month_len=month_len,
)

    # 4) Executa apenas o bloco HAR
    df_train, df_test = _run_best_har_block(
        df_train=df_train,
        df_test=df_test,
        best_rules_number_har=best_rules_number_har,
        best_ze_har=best_ze_har,
        lamb=lamb,
        alfa=alfa,
        week_len=week_len,
        month_len=month_len,
    )

    # 5) Salva outputs
    train_out_path, test_out_path = _save_outputs(
        df_train=df_train,
        df_test=df_test,
        symbol=symbol,
        horizon=horizon,
        symbol_out_dir=paths["symbol_out_dir"],
    )

    print(
        f"[{symbol} | t+{horizon} | HAR only] concluído.\n"
        f"Train: {train_out_path}\n"
        f"Test:  {test_out_path}\n"
        f"Best HAR: r={best_rules_number_har}, ze={best_ze_har}"
    )

    return df_train, df_test

In [28]:
#symbols = ["ADAUSDT", "BNBUSDT", "BTCUSDT", "DOGEUSDT", "ETHUSDT", "TRXUSDT", "XLMUSDT", "XRPUSDT"]
symbols = ["BTCUSDT", "ETHUSDT"]

all_results = {}

for horizon in [1, 7, 30]:
    all_results[horizon] = {}
    for sym in symbols:
        df_tr, df_te = run_fuzzy_for_symbol(sym, horizon)
        all_results[horizon][sym] = {"train": df_tr, "test": df_te}

[BTCUSDT | t+1 | HAR only] concluído.
Train: /content/drive/MyDrive/ProductionFuzzyHARPipeline/RV_t+1/BTCUSDT/BTCUSDT_tplus1_train_with_fuzzy.xlsx
Test:  /content/drive/MyDrive/ProductionFuzzyHARPipeline/RV_t+1/BTCUSDT/BTCUSDT_tplus1_test_with_fuzzy.xlsx
Best HAR: r=6, ze=1.0
[ETHUSDT | t+1 | HAR only] concluído.
Train: /content/drive/MyDrive/ProductionFuzzyHARPipeline/RV_t+1/ETHUSDT/ETHUSDT_tplus1_train_with_fuzzy.xlsx
Test:  /content/drive/MyDrive/ProductionFuzzyHARPipeline/RV_t+1/ETHUSDT/ETHUSDT_tplus1_test_with_fuzzy.xlsx
Best HAR: r=6, ze=1.0
[BTCUSDT | t+7 | HAR only] concluído.
Train: /content/drive/MyDrive/ProductionFuzzyHARPipeline/RV_t+7/BTCUSDT/BTCUSDT_tplus7_train_with_fuzzy.xlsx
Test:  /content/drive/MyDrive/ProductionFuzzyHARPipeline/RV_t+7/BTCUSDT/BTCUSDT_tplus7_test_with_fuzzy.xlsx
Best HAR: r=6, ze=1.0
[ETHUSDT | t+7 | HAR only] concluído.
Train: /content/drive/MyDrive/ProductionFuzzyHARPipeline/RV_t+7/ETHUSDT/ETHUSDT_tplus7_train_with_fuzzy.xlsx
Test:  /content/drive/

# Results